In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [4]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 6.8",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [5]:
for runs in runs:
    print(f"run id: {runs.info.run_id}, rmse: {runs.data.metrics['rmse']:.4f}")

run id: 907b2602943d4e5aaf8a92e2a9d30b31, rmse: 6.3184
run id: c3bc486d4b9343a696ac9a394b3c6436, rmse: 6.3184
run id: 95969795ad8c413eb3c1d1e0177c689e, rmse: 6.3335
run id: 07efb31a1cdf4cd39c9ae96a90331566, rmse: 6.3662
run id: 4d8f0059ae274bffb95c5aed869ef3b1, rmse: 6.3759


In [6]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [7]:
run_id = "4d8f0059ae274bffb95c5aed869ef3b1"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/03/15 01:07:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 4d8f0059ae274bffb95c5aed869ef3b1 has no artifacts at artifact path 'model', registering model based on models:/m-ec0deeb89ab1481e9a75b5febdb3961a instead
Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1773529651453, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1773529651453, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='4d8f0059ae274bffb95c5aed869ef3b1', run_link=None, source='models:/m-ec0deeb89ab1481e9a75b5febdb3961a', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [8]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)


for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 4, stage: None
version: 3, stage: Staging


C:\Users\mae25\AppData\Local\Temp\ipykernel_10476\3828318741.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [9]:
client.transition_model_version_stage(
    name=model_name,
    version=3,
    stage="Staging",
    archive_existing_versions=False
)

C:\Users\mae25\AppData\Local\Temp\ipykernel_10476\3248838605.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773526997706, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1773529652545, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='4d8f0059ae274bffb95c5aed869ef3b1', run_link=None, source='models:/m-ec0deeb89ab1481e9a75b5febdb3961a', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

In [28]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)



def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [13]:
df = read_dataframe("../data/green_tripdata_2021-03.parquet")

In [21]:
client.download_artifacts(run_id="907b2602943d4e5aaf8a92e2a9d30b31", path='preprocessor', dst_path='.')

'a:\\AI\\mlops\\experiment-tracking\\preprocessor'

In [22]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [23]:
X_test = preprocess(df, dv)

In [24]:
target = "duration"
y_test = df[target].values

In [29]:
%time test_model(name=model_name, stage=None, X_test=X_test, y_test=y_test)

CPU times: total: 9.08 s
Wall time: 4.49 s


{'rmse': 6.315779064792659}

In [30]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="Production",
    archive_existing_versions=True
)

C:\Users\mae25\AppData\Local\Temp\ipykernel_10476\1703833061.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773524550925, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1773530218174, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='907b2602943d4e5aaf8a92e2a9d30b31', run_link='', source='models:/m-a186309219d44e0abb49906ba3161b33', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>